# Preprocessing & Adding Features

#### INPUT  
- filtered_dataset.tsv

#### WHAT THIS NOTEBOOK DOES
- loads the .tsv file filtered with notebook 01_load.ipynb
- merges subhead with content where applicable
- deletes unneccessary columns (including the now-merged subhead)
- removes HTML tags from content
- adds columns for additional features: publisher, sentiment_score, emotion scores (anger, fear, joy, sadness), linguistic_complexity
- writes the preprocessed dataframe into a new .tsv file

#### OUTPUTS 
- preprocessed_dataset.tsv


In [ ]:
# imports

import re

from config import preprocess_config as cfg
from lib import preprocess_lib as lib

import pandas as pd


In [ ]:
df_filtered = pd.read_csv(cfg.INPUT_TSV, sep="\t")

print(f"Loaded {len(df_filtered)} articles from {cfg.INPUT_TSV}")

df_filtered.head()

In [ ]:
# merge subhead with content if subhead is not null (done before dropping the subhead column)
df_filtered["content"] = df_filtered.apply(lambda row: row["subhead"] + " " + row["content"] if pd.notnull(row["subhead"]) else row["content"], axis=1)

# print full content of one sample article where subhead is not null to check that the merge worked correctly
print(df_filtered.loc[df_filtered["subhead"].notna(), "content"].sample(1).iloc[0])


In [ ]:
# delete columns that are not needed for the analysis (subhead is dropped here after being merged into content)
columns_to_drop = cfg.COLUMNS_TO_DROP
df_filtered = df_filtered.drop(columns=columns_to_drop)

df_filtered.head()


In [ ]:
# remove html tags from content column

def remove_html_tags(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, ' ', text)

def remove_double_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

df_filtered["content"] = df_filtered["content"].apply(remove_html_tags)
df_filtered["content"] = df_filtered["content"].apply(remove_double_spaces)

# print full content of one sample article to check that the cleaning worked correctly
print(df_filtered["content"].sample(1).iloc[0])


In [ ]:
# add publisher column by mapping medium_code to publisher using the config file dictionary

df_filtered["publisher"] = df_filtered["medium_code"].map(cfg.MEDIUM_CODE_TO_PUBLISHER)

df_filtered.groupby("publisher").size()

In [ ]:
# add a sentiment score column (positive - negative probability) using the German sentiment model

df_filtered["sentiment_score"] = lib.score_sentiment(df_filtered)

df_filtered[["content", "sentiment_score"]].head()


In [ ]:
# add emotion score columns (anger, fear, joy, sadness) using the multilingual emotion model

anger, joy, sadness, fear = lib.score_emotions(df_filtered)
df_filtered["emotion_anger"] = anger
df_filtered["emotion_fear"] = fear
df_filtered["emotion_joy"] = joy
df_filtered["emotion_sadness"] = sadness

df_filtered[["content", "emotion_anger", "emotion_fear", "emotion_joy", "emotion_sadness"]].head()


In [ ]:
# add a linguistic complexity score column (German Flesch reading ease; higher = simpler)

df_filtered["linguistic_complexity"] = df_filtered["content"].apply(lib.flesch_reading_ease_de)

df_filtered[["content", "linguistic_complexity"]].head()


In [ ]:
# write the preprocessed dataframe to a new .tsv file

df_filtered.to_csv(cfg.OUTPUT_TSV, sep="\t", index=False)

print(f"Wrote {len(df_filtered)} preprocessed articles to {cfg.OUTPUT_TSV}")
